# E12-K1: FP8 kernel probe for Qwen3.8-27B on 4×L4 (no weight change)

**Why:** R5's log shows that all 5 main block-FP8 GEMM shapes of our model run vLLM's Triton kernel with the **default config** on L4 ("Performance might be sub-optimal!"). vLLM 0.19 has no L4 config files for them. FP8 Marlin (weight-only, block-128 supported) is off on sm89 unless `VLLM_TEST_FORCE_FP8_MARLIN=1`.

**Stages:**
1. Install vLLM 0.19.0 from R5's wheelhouse, plus R5's libcuda shim. The code is copied from R5.
2. Tune the 5 shapes in parallel on the 4 GPUs with vLLM's own kernel. Decode sizes go first, under a time budget, and each best config is checked against the default output.
3. Engine benchmark, one fresh process per variant, all with R5's exact engine settings (TP4, fp8 KV, MTP-3, FLASHINFER, prefix caching, 16 seqs):
   - **default**;
   - **tuned** (L4 config files copied into vLLM);
   - **marlin** (`VLLM_TEST_FORCE_FP8_MARLIN=1`).
4. Report:
   - decode tok/s at about 2–10k and about 36k context;
   - MTP acceptance and preemptions;
   - which kernel path the logs show;
   - greedy token agreement against default.

**Run:** 4×L4, Internet OFF, the pinned Q27 Docker image. Attach the FP8 model, R5's vLLM wheelhouse and the competition data. Expected time: about 1.5–2 h; a guard stops new stages after `K1_BUDGET_SEC` (default 4 h).

**Outputs:** `/kaggle/working/k1/`:
- `results.json`, `report.md`;
- `configs_l4/*.json`: tuned configs, ready to ship;
- `timings_*.json`, `tune_gpu*.log`, `bench_*.json` / `.log`.

This is not a submission.


In [ ]:
# ==================================================================== CELL 1 — setup (stdlib only)
import os, sys, json, time, subprocess, shutil, glob, importlib.util
from pathlib import Path
T0 = time.time()
DRYRUN = os.environ.get("K1_DRYRUN") == "1"          # local CPU orchestration test only
WORKING = Path(os.environ.get("K1_WORKING", "/kaggle/working")); OUT = WORKING / "k1"; OUT.mkdir(parents=True, exist_ok=True)
MODEL_PATH = Path(os.environ.get("Q27X_MODEL", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"))
WHEELHOUSE = Path(os.environ.get("Q27X_WHEELHOUSE", "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"))
ARC_ROOT = Path(os.environ.get("Q27X_ARC_ROOT", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"))
if not ARC_ROOT.exists():
    for _alt in ("/kaggle/input/arc-prize-2026-arc-agi-2",):
        if Path(_alt).exists():
            ARC_ROOT = Path(_alt)
os.environ["Q27X_MODEL"] = str(MODEL_PATH); os.environ["Q27X_ARC_ROOT"] = str(ARC_ROOT)
TIME_BUDGET_SEC = float(os.environ.get("K1_BUDGET_SEC", 4 * 3600))      # no new stage after this
TUNE_SEC = float(os.environ.get("K1_TUNE_SEC", 40 * 60))                 # per-GPU tuning budget
BENCH_TIMEOUT_SEC = float(os.environ.get("K1_BENCH_TIMEOUT_SEC", 45 * 60))
# (N, K) per TP4 rank: the 5 shapes of R5's "Using default W8A8 Block FP8 kernel config" warnings
SHAPES = [(8704, 5120), (5120, 4352), (5120, 1536), (4096, 5120), (3584, 5120)]
GPU_SHAPES = [[(8704, 5120)], [(5120, 4352), (5120, 1536)], [(4096, 5120)], [(3584, 5120)]]   # balanced by size
RESULTS = {"started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "dryrun": DRYRUN, "stages": {}}

def save_results():
    (OUT / "results.json").write_text(json.dumps(RESULTS, indent=1, default=str), encoding="utf-8")

def time_left():
    return TIME_BUDGET_SEC - (time.time() - T0)

def sh(cmd, timeout=60):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout).stdout
    except Exception as e:
        return f"ERR {e!r}"

print("K1 setup | dryrun", DRYRUN, "| model", MODEL_PATH, "| arc", ARC_ROOT, "| out", OUT)
RESULTS["nvidia_smi"] = sh(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
print(RESULTS["nvidia_smi"]); save_results()


In [ ]:
# ==================================================================== CELL 2 — offline install (R5's code) + libcuda shim (R5's code)
if DRYRUN:
    os.environ["Q27X_SKIP_INSTALL"] = "1"
import importlib, importlib.metadata
def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

EXPECTED_RUNTIME_PACKAGES = {"vllm": "0.19.0", "transformers": "4.57.6", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2"}
_loaded = {"vllm": "vllm", "transformers": "transformers", "tokenizers": "tokenizers", "huggingface-hub": "huggingface_hub"}
before_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}

def offline_pip_install(requirements):
    if not requirements: return
    if not WHEELHOUSE.exists():
        raise FileNotFoundError(f"wheelhouse not attached: {WHEELHOUSE}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])

if os.environ.get("Q27X_SKIP_INSTALL") != "1":     # Q27X_SKIP_INSTALL is for the local harness only
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(f"Python {sys.version.split()[0]}: the cp312 wheelhouse needs the pinned Python 3.12 Kaggle image")
    for _pkg, _mod in _loaded.items():
        if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _mod in sys.modules:
            raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart the session before Run All.")
    if before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        offline_pip_install(["vllm==0.19.0"])
    else:
        offline_pip_install([f"{p}=={EXPECTED_RUNTIME_PACKAGES[p]}" for p in ("transformers", "tokenizers", "huggingface-hub")
                             if before_stack[p] != EXPECTED_RUNTIME_PACKAGES[p]])
    after_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}
    for _p, _want in EXPECTED_RUNTIME_PACKAGES.items():
        if after_stack.get(_p) != _want:
            raise RuntimeError(f"runtime parity failure: {_p}={after_stack.get(_p)!r}, expected {_want!r}")
    print("runtime parity PASS:", after_stack)

if not DRYRUN:
    import shutil as _shutil
    def ensure_libcuda_linkable(candidates=None, workdir="/tmp/q27x_libcuda_shim"):
        info = {"needed": None, "real": None, "shim": None, "probe": "FAIL"}
        try:
            work = Path(workdir); work.mkdir(parents=True, exist_ok=True)
            if candidates is None:
                existing = [Path(d, "libcuda.so") for d in ("/usr/local/cuda/lib64", "/usr/local/cuda/lib64/stubs",
                                                            "/usr/lib/x86_64-linux-gnu", "/usr/lib64")]
                existing = [p for p in existing if p.exists()]
            else:
                existing = []
            if existing:
                info.update(needed=False, real=str(existing[0]))
            else:
                cands = list(candidates) if candidates is not None else []
                if candidates is None:
                    for d in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64",
                              "/usr/local/cuda/compat"):
                        try:
                            cands += sorted(str(p) for p in Path(d).glob("libcuda.so.*") if p.is_file())
                        except Exception:
                            pass
                    try:
                        for line in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, timeout=30).stdout.splitlines():
                            if "libcuda.so.1" in line and "=>" in line:
                                cands.append(line.split("=>")[-1].strip())
                    except Exception:
                        pass
                real = next((Path(c) for c in cands if Path(c).is_file()), None)
                if real is None:
                    info["reason"] = "libcuda.so.1 not found"
                    return info
                link = work / "libcuda.so"
                if link.exists() or link.is_symlink():
                    link.unlink()
                try:
                    link.symlink_to(real.resolve())
                except OSError:
                    _shutil.copyfile(real, link)
                old = os.environ.get("LIBRARY_PATH", "")
                os.environ["LIBRARY_PATH"] = f"{work}:{old}" if old else str(work)
                info.update(needed=True, real=str(real), shim=str(link), library_path=os.environ["LIBRARY_PATH"])
            cxx = _shutil.which("c++") or _shutil.which("g++")
            if not cxx:
                info["reason"] = "no C++ compiler for the link probe"
                return info
            cp = subprocess.run([cxx, "-shared", "-fPIC", "-x", "c++", "-", "-lcuda", "-o", str(work / "probe.so")],
                                input='extern "C" int q27x_probe() { return 0; }\n', text=True, capture_output=True,
                                timeout=120, env=os.environ.copy())
            info["probe"] = "PASS" if cp.returncode == 0 else "FAIL"
            if cp.returncode != 0:
                info["probe_stderr"] = (cp.stderr or "")[-1500:]
        except Exception as e:
            info["reason"] = f"{type(e).__name__}: {e}"[:500]
        return info

    os.environ.setdefault("MAX_JOBS", "4")
    CUDA_LINK = ensure_libcuda_linkable()
else:
    CUDA_LINK = {"dryrun": True}
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
RESULTS["stages"]["install"] = {"cuda_link": CUDA_LINK, "sec": round(time.time() - T0, 1)}
print("CUDA/FlashInfer linker preflight:", json.dumps(CUDA_LINK)); save_results()


In [ ]:
# ==================================================================== CELL 3 — stage 2: tune the 5 block-FP8 shapes (4 GPUs in parallel)
TUNER_SRC = '"""E12-K1 tuner: search W8A8 block-FP8 Triton configs on one L4 for our Qwen3.8-27B TP4 GEMM shapes.\n\nIt uses vLLM 0.19.0\'s own production kernel (`_w8a8_triton_block_scaled_mm`) with the same call as\n`w8a8_triton_block_scaled_mm`. The search follows vLLM\'s benchmarks/kernels/benchmark_w8a8_block_fp8.py (Apache-2.0),\npruned per batch size so that it fits the time budget. For every (shape, M) it records:\n- the default-config time;\n- the best time and config;\n- the max relative error of the best config against the default.\n\nIt writes vLLM\'s config file name: N=..,K=..,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128,128].json.\n\nusage: python k1_tune_block_fp8.py --shapes 8704x5120,4096x5120 --out-dir DIR --deadline UNIX_TS --label gpu0\nK1_DRYRUN=1 writes fake results without torch/vLLM (local orchestration test only).\n"""\nimport argparse, json, os, random, sys, time\n\nap = argparse.ArgumentParser()\nap.add_argument("--shapes", required=True)          # "N1xK1,N2xK2"\nap.add_argument("--out-dir", required=True)\nap.add_argument("--deadline", type=float, required=True)\nap.add_argument("--label", default="gpu")\nargs = ap.parse_args()\nSHAPES = [tuple(int(v) for v in s.split("x")) for s in args.shapes.split(",") if s]\nOUT = args.out_dir\nos.makedirs(OUT, exist_ok=True)\n# decode first: 16 seqs x (1 + 3 MTP) = 64 tokens per verify step, 16 per MTP draft step; then prefill chunks (4096 max)\nM_ORDER = [64, 16, 48, 32, 96, 128, 4096, 2048, 1024, 512, 256, 24, 8, 1536, 3072]\nBLOCK = [128, 128]\nDEFAULT = {"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 128, "GROUP_SIZE_M": 32, "num_warps": 4, "num_stages": 2}\n\n\ndef search_space(M):\n    bms = [16, 32, 64] if M <= 128 else [64, 128, 256]\n    groups = [1] if M <= 128 else [1, 16, 32]\n    out = []\n    for st in (2, 3, 4, 5):\n        for bm in bms:\n            for bk in (64, 128):\n                for bn in (32, 64, 128, 256):\n                    for nw in (4, 8):\n                        for g in groups:\n                            out.append({"BLOCK_SIZE_M": bm, "BLOCK_SIZE_N": bn, "BLOCK_SIZE_K": bk, "GROUP_SIZE_M": g,\n                                        "num_warps": nw, "num_stages": st})\n    # a different order per GPU process: parallel processes then compile different configs first and reuse each\n    # other\'s results through Triton\'s shared disk cache (the best config does not depend on the order)\n    random.Random(f"{args.label}-{M}").shuffle(out)\n    return out\n\n\ndef file_name(N, K, device_name):\n    return f"N={N},K={K},device_name={device_name},dtype=fp8_w8a8,block_shape=[{BLOCK[0]},{BLOCK[1]}].json"\n\n\ndef log(*a):\n    print(f"[{args.label} {time.strftime(\'%H:%M:%S\')}]", *a, flush=True)\n\n\nif os.environ.get("K1_DRYRUN") == "1":\n    for (N, K) in SHAPES:\n        cfg = {str(M): dict(DEFAULT, num_stages=3) for M in M_ORDER[:6]}\n        json.dump(cfg, open(os.path.join(OUT, file_name(N, K, "NVIDIA_L4")), "w"), indent=1)\n        tim = {str(M): {"default_us": 100.0, "best_us": 80.0, "speedup": 1.25, "max_rel_err": 0.0, "configs_tried": 3}\n               for M in M_ORDER[:6]}\n        json.dump({"N": N, "K": K, "timings": tim, "complete": False}, open(os.path.join(OUT, f"timings_N{N}_K{K}.json"), "w"))\n    log("DRYRUN done", SHAPES)\n    sys.exit(0)\n\nimport torch\nfrom vllm.model_executor.layers.quantization.utils.fp8_utils import _w8a8_triton_block_scaled_mm\nfrom vllm.platforms import current_platform\nfrom vllm.triton_utils import triton\n\nDEVICE_NAME = current_platform.get_device_name().replace(" ", "_")\nOUT_DTYPE = torch.bfloat16          # the model runs in bf16, so the block-FP8 GEMM output is bf16\n\n\ndef mm(A, B, As, Bs, cfg):\n    M = A.numel() // A.shape[-1]\n    N, K = B.shape\n    C = A.new_empty(A.shape[:-1] + (N,), dtype=OUT_DTYPE)\n    grid = lambda META: (triton.cdiv(M, META["BLOCK_SIZE_M"]) * triton.cdiv(N, META["BLOCK_SIZE_N"]),)\n    _w8a8_triton_block_scaled_mm[grid](A, B, C, As, Bs, M, N, K, BLOCK[0], BLOCK[1],\n                                       A.stride(-2), A.stride(-1), B.stride(1), B.stride(0), C.stride(-2), C.stride(-1),\n                                       As.stride(-2), As.stride(-1), Bs.stride(1), Bs.stride(0), **cfg)\n    return C\n\n\ndef bench(A, B, As, Bs, cfg, iters=20):\n    for _ in range(3):\n        mm(A, B, As, Bs, cfg)\n    torch.cuda.synchronize()\n    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)\n    s.record()\n    for _ in range(iters):\n        mm(A, B, As, Bs, cfg)\n    e.record(); e.synchronize()\n    return s.elapsed_time(e) * 1000.0 / iters          # microseconds per call\n\n\ndef inputs(M, N, K):\n    fi = torch.finfo(torch.float8_e4m3fn)\n    A = ((torch.rand(M, K, device="cuda") - 0.5) * 2 * fi.max).clamp(fi.min, fi.max).to(torch.float8_e4m3fn)\n    B = ((torch.rand(N, K, device="cuda") - 0.5) * 2 * fi.max).clamp(fi.min, fi.max).to(torch.float8_e4m3fn).contiguous()\n    As = torch.rand(M, triton.cdiv(K, BLOCK[1]), device="cuda") * 1e-2\n    Bs = torch.rand(triton.cdiv(N, BLOCK[0]), triton.cdiv(K, BLOCK[1]), device="cuda") * 1e-2\n    return A, B, As, Bs\n\n\nlog("device", DEVICE_NAME, "shapes", SHAPES, "deadline in", round(args.deadline - time.time()), "s")\nfor (N, K) in SHAPES:\n    best_cfgs, timings = {}, {}\n    complete = True\n    for M in M_ORDER:\n        if time.time() > args.deadline:\n            complete = False; log("deadline: stop before M", M); break\n        A, B, As, Bs = inputs(M, N, K)\n        t_def = bench(A, B, As, Bs, DEFAULT)\n        ref = mm(A, B, As, Bs, DEFAULT).float()\n        best, best_t, tried, cut = DEFAULT, t_def, 0, False\n        for cfg in search_space(M):\n            if time.time() > args.deadline:\n                cut = True; break\n            try:\n                t = bench(A, B, As, Bs, cfg)\n            except Exception:          # OutOfResources / compile errors for oversized tiles\n                continue\n            tried += 1\n            if t < best_t:\n                best, best_t = cfg, t\n        out = mm(A, B, As, Bs, best).float()\n        rel = ((out - ref).abs().max() / (ref.abs().max() + 1e-9)).item()\n        if rel > 1e-2:                     # a wrong kernel must never be written\n            log(f"N={N} K={K} M={M}: best config differs from default by {rel:.3g}; keeping default")\n            best, best_t = DEFAULT, t_def\n        best_cfgs[str(M)] = best\n        timings[str(M)] = {"default_us": round(t_def, 2), "best_us": round(best_t, 2), "speedup": round(t_def / best_t, 3),\n                           "max_rel_err": rel, "configs_tried": tried, "cut_by_deadline": cut}\n        log(f"N={N} K={K} M={M}: default {t_def:.1f} us -> best {best_t:.1f} us (x{t_def / best_t:.2f}), tried {tried}")\n        del A, B, As, Bs, ref, out\n        if cut:\n            complete = False; break\n    if best_cfgs:\n        with open(os.path.join(OUT, file_name(N, K, DEVICE_NAME)), "w") as f:\n            json.dump(best_cfgs, f, indent=1); f.write("\\n")\n    with open(os.path.join(OUT, f"timings_N{N}_K{K}.json"), "w") as f:\n        json.dump({"N": N, "K": K, "device": DEVICE_NAME, "timings": timings, "complete": complete}, f, indent=1)\nlog("done")\n'
(OUT / "k1_tune_block_fp8.py").write_text(TUNER_SRC, encoding="utf-8")
TUNE_DIR = OUT / "tune"; TUNE_DIR.mkdir(exist_ok=True)
_t = time.time(); _procs = []
_budget = max(5 * 60, min(TUNE_SEC, time_left() - 75 * 60))     # keep at least 75 min for the engine benchmarks
_deadline = time.time() + _budget
for _gpu, _shapes in enumerate(GPU_SHAPES):
    _env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(_gpu))
    _log = open(OUT / f"tune_gpu{_gpu}.log", "w")
    _cmd = [sys.executable, str(OUT / "k1_tune_block_fp8.py"), "--shapes", ",".join(f"{n}x{k}" for n, k in _shapes),
            "--out-dir", str(TUNE_DIR), "--deadline", str(_deadline), "--label", f"gpu{_gpu}"]
    _procs.append((_gpu, subprocess.Popen(_cmd, env=_env, stdout=_log, stderr=subprocess.STDOUT), _log))
_rc = {}
for _gpu, _p, _log in _procs:
    try:
        _rc[_gpu] = _p.wait(timeout=max(60, _deadline - time.time() + 10 * 60))
    except subprocess.TimeoutExpired:
        _p.kill(); _rc[_gpu] = "killed_after_deadline"
    _log.close()
CONFIG_DIR = OUT / "configs_l4"; CONFIG_DIR.mkdir(exist_ok=True)
for _f in TUNE_DIR.glob("N=*,K=*,device_name=*.json"):
    shutil.copy(_f, CONFIG_DIR / _f.name)
TUNE_TIMINGS = {}
for _f in sorted(TUNE_DIR.glob("timings_*.json")):
    _d = json.loads(_f.read_text()); TUNE_TIMINGS[f"N={_d['N']},K={_d['K']}"] = _d
RESULTS["stages"]["tune"] = {"sec": round(time.time() - _t, 1), "budget_sec": round(_budget), "returncodes": _rc,
                             "config_files": sorted(p.name for p in CONFIG_DIR.glob("*.json")), "timings": TUNE_TIMINGS}
for _k, _d in TUNE_TIMINGS.items():
    print(_k, "complete" if _d.get("complete") else "PARTIAL", {m: v["speedup"] for m, v in _d["timings"].items()})
save_results()


In [ ]:
# ==================================================================== CELL 4 — stage 3: engine benchmark per variant (fresh process each)
BENCH_SRC = '"""E12-K1 engine benchmark: one vLLM 0.19 engine (R5\'s exact settings) in this process; decode speed with paired workloads.\n\nThe variant is chosen by the parent before this process starts:\n- default: no L4 config files, Triton block FP8 with the default config;\n- tuned: L4 config files copied into vLLM\'s configs directory;\n- marlin: VLLM_TEST_FORCE_FP8_MARLIN=1.\n\nWorkloads (prefix caching ON, so each *_prefill run warms the cache and the next run measures mostly decode):\n- short_prefill / short: 16 ARC prompts (about 2-10k tokens); 3,072 new tokens each; T 1.0, top_p 0.95, top_k 20; ignore_eos.\n- long_prefill / long: 16 prompts of about 36k tokens; 1,024 new tokens each.\n- greedy: 4 short prompts, 192 greedy tokens, compared against the default variant by the parent.\n\nusage: python k1_engine_bench.py --variant default --out result.json\nK1_DRYRUN=1 writes fake results (local orchestration test only).\n"""\nimport argparse, json, os, sys, time, traceback\n\nap = argparse.ArgumentParser()\nap.add_argument("--variant", required=True)\nap.add_argument("--out", required=True)\nargs = ap.parse_args()\nMODEL = os.environ.get("Q27X_MODEL", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1")\nARC_ROOT = os.environ.get("Q27X_ARC_ROOT", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2")\nRESULT = {"variant": args.variant, "errors": [], "workloads": {}, "env_force_fp8_marlin": os.environ.get("VLLM_TEST_FORCE_FP8_MARLIN")}\n\n\ndef save():\n    with open(args.out, "w") as f:\n        json.dump(RESULT, f, indent=1)\n\n\nif os.environ.get("K1_DRYRUN") == "1":\n    f = {"default": 1.0, "tuned": 1.15, "marlin": 1.25}.get(args.variant, 1.0)\n    RESULT.update(load_sec=1.0, kv_tokens=1353600)\n    for w, base in (("short", 300.0), ("long", 240.0)):\n        RESULT["workloads"][w] = {"wall_sec": 10.0, "gen_tokens": 1000, "tok_s": base * f, "accept_rate": 0.55, "preemptions": 0}\n        RESULT["workloads"][w + "_prefill"] = {"wall_sec": 5.0, "prompt_tokens": 50000, "prefill_tok_s": 4000.0 * f}\n    RESULT["greedy_ids"] = [[1, 2, 3, 4, 5]] * 4 if args.variant != "marlin" else [[1, 2, 3, 9, 9]] * 4\n    print("DRYRUN", args.variant); save(); sys.exit(0)\n\nos.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\nos.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"\nos.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"\nfrom transformers import AutoTokenizer\nfrom vllm import LLM, SamplingParams\n\ntok = AutoTokenizer.from_pretrained(MODEL)\nch = json.load(open(os.path.join(ARC_ROOT, "arc-agi_evaluation_challenges.json")))\nTIDS = sorted(ch)\nINSTR = ("You are solving an ARC-AGI task. Study the training pairs, find the transformation rule, then write a Python "\n         "function transform(grid) that maps each input grid to its output grid.\\n\\n")\n\n\ndef render(task_texts):\n    msg = INSTR + "\\n\\n".join(task_texts)\n    return tok.apply_chat_template([{"role": "user", "content": msg}], tokenize=False, add_generation_prompt=True,\n                                   enable_thinking=True, reasoning_effort="xhigh")\n\n\ndef ids_of(text):\n    return tok(text, add_special_tokens=False)["input_ids"]\n\n\nshort = [ids_of(render([json.dumps(ch[t])])) for t in TIDS[:16]]\nlong_ = []\nfor i in range(16):        # whole tasks until the next one would pass 36k tokens, so the chat template ends correctly\n    parts, j, ids = [], i * 7, None\n    while True:\n        cand = ids_of(render(parts + [json.dumps(ch[TIDS[j % len(TIDS)]])])); j += 1\n        if len(cand) > 36000 and ids is not None:\n            break\n        parts.append(json.dumps(ch[TIDS[(j - 1) % len(TIDS)]])); ids = cand\n        if len(ids) > 36000 or len(parts) > 60:\n            break\n    long_.append(ids)\nRESULT["prompt_tokens"] = {"short_mean": sum(map(len, short)) / 16, "long_mean": sum(map(len, long_)) / 16}\n\nt0 = time.time()\nllm = LLM(model=MODEL, tensor_parallel_size=4, distributed_executor_backend="mp", dtype="auto", load_format="safetensors",\n          trust_remote_code=False, language_model_only=True, max_model_len=106496, max_num_seqs=16,\n          max_num_batched_tokens=4096, gpu_memory_utilization=0.90, enforce_eager=False, disable_custom_all_reduce=True,\n          enable_prefix_caching=True, seed=20261005, disable_log_stats=False, kv_cache_dtype="fp8_e4m3",\n          speculative_config={"method": "mtp", "num_speculative_tokens": 3}, attention_config={"backend": "FLASHINFER"})\nRESULT["load_sec"] = round(time.time() - t0, 1)\nprint("ENGINE READY", args.variant, RESULT["load_sec"], "s", flush=True)\nsave()\n\n\ndef counters():\n    out = {}\n    try:\n        for m in llm.get_metrics():\n            v = getattr(m, "value", None)\n            if isinstance(v, (int, float)):\n                out[m.name] = out.get(m.name, 0) + v\n    except Exception as e:\n        RESULT["errors"].append(f"metrics: {e!r}"[:300])\n    return out\n\n\ndef run(name, prompts, max_tokens, temperature=1.0, ignore_eos=True):\n    kw = {"top_p": 0.95, "top_k": 20} if temperature > 0 else {}      # greedy: library defaults\n    sps = [SamplingParams(temperature=temperature, max_tokens=max_tokens, ignore_eos=ignore_eos, seed=1000 + i, **kw)\n           for i in range(len(prompts))]\n    c0 = counters(); t = time.time()\n    outs = llm.generate([{"prompt_token_ids": p} for p in prompts], sps, use_tqdm=False)\n    wall = time.time() - t; c1 = counters()\n    d = lambda k: c1.get(k, 0) - c0.get(k, 0)\n    gen = sum(len(o.outputs[0].token_ids) for o in outs)\n    draft, acc = d("vllm:spec_decode_num_draft_tokens"), d("vllm:spec_decode_num_accepted_tokens")\n    rec = {"wall_sec": round(wall, 2), "gen_tokens": gen, "tok_s": round(gen / wall, 1),\n           "prompt_tokens": int(d("vllm:prompt_tokens")), "prompt_cached": int(d("vllm:prompt_tokens_cached")),\n           "prefill_tok_s": round(d("vllm:prompt_tokens") / wall, 1) if max_tokens == 1 else None,\n           "accept_rate": round(acc / draft, 4) if draft else None,\n           "tokens_per_step": round(1 + acc / d("vllm:spec_decode_num_drafts"), 3) if d("vllm:spec_decode_num_drafts") else None,\n           "preemptions": int(d("vllm:num_preemptions"))}\n    RESULT["workloads"][name] = rec\n    print("WORKLOAD", args.variant, name, json.dumps(rec), flush=True)\n    save()\n    return outs\n\n\nfor name, fn in (("short_prefill", lambda: run("short_prefill", short, 1)),\n                 ("short", lambda: run("short", short, 3072)),\n                 ("long_prefill", lambda: run("long_prefill", long_, 1)),\n                 ("long", lambda: run("long", long_, 1024))):\n    try:\n        fn()\n    except Exception:\n        RESULT["errors"].append(f"{name}: " + traceback.format_exc()[-1500:]); save()\ntry:\n    g = run("greedy", short[:4], 192, temperature=0.0)\n    RESULT["greedy_ids"] = [list(o.outputs[0].token_ids) for o in g]\n    RESULT["greedy_text_head"] = [tok.decode(o.outputs[0].token_ids[:60]) for o in g]\nexcept Exception:\n    RESULT["errors"].append("greedy: " + traceback.format_exc()[-1500:])\nsave()\nprint("BENCH DONE", args.variant, flush=True)\n'
(OUT / "k1_engine_bench.py").write_text(BENCH_SRC, encoding="utf-8")
if DRYRUN:
    VLLM_DIR = Path(os.environ["K1_VLLM_DIR"])
else:
    VLLM_DIR = Path(importlib.util.find_spec("vllm").submodule_search_locations[0])
VLLM_CFG_DIR = VLLM_DIR / "model_executor" / "layers" / "quantization" / "utils" / "configs"
OUR_CFGS = sorted(CONFIG_DIR.glob("*.json"))

def set_configs(install):
    for f in OUR_CFGS:
        dst = VLLM_CFG_DIR / f.name
        if install:
            shutil.copy(f, dst)
        elif dst.exists():
            dst.unlink()
    return sorted(p.name for p in VLLM_CFG_DIR.glob("*device_name=NVIDIA_L4*fp8_w8a8*"))

def kill_children():
    try:
        import psutil
        for k in psutil.Process(os.getpid()).children(recursive=True):
            try: k.kill()
            except Exception: pass
    except Exception:
        pass

def wait_gpu_free(max_sec=180):
    if DRYRUN:
        return True
    t = time.time()
    while time.time() - t < max_sec:
        try:
            used = [int(x) for x in sh(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"]).split()]
            if used and max(used) < 1024:
                return True
        except Exception:
            pass
        time.sleep(5)
    return False

def log_evidence(text):
    return {"default_block_fp8_warnings": text.count("Using default W8A8 Block FP8 kernel config"),
            "tuned_config_used": text.count("for W8A8 Block FP8 kernel"),
            "marlin_mentions": text.count("Marlin"),
            "selected_lines": sorted({l.strip()[-110:] for l in text.splitlines() if "Selected" in l and "Kernel" in l})[:6]}

VARIANTS = [("default", False, {}), ("tuned", True, {}), ("marlin", False, {"VLLM_TEST_FORCE_FP8_MARLIN": "1"})]
BENCH = {}
for _name, _install, _extra in VARIANTS:
    if _name == "tuned" and not OUR_CFGS:
        BENCH[_name] = {"skipped": "no tuned config files"}; continue
    if time_left() < 25 * 60:
        BENCH[_name] = {"skipped": f"time guard ({round(time_left() / 60)} min left)"}; continue
    _installed = set_configs(_install)
    _env = dict(os.environ, **_extra)
    _out = OUT / f"bench_{_name}.json"; _logp = OUT / f"bench_{_name}.log"
    _t = time.time()
    with open(_logp, "w") as _log:
        try:
            _rc = subprocess.run([sys.executable, str(OUT / "k1_engine_bench.py"), "--variant", _name, "--out", str(_out)],
                                 env=_env, stdout=_log, stderr=subprocess.STDOUT, timeout=BENCH_TIMEOUT_SEC).returncode
        except subprocess.TimeoutExpired:
            _rc = "timeout"
    kill_children(); _free = wait_gpu_free()
    _res = json.loads(_out.read_text()) if _out.exists() else {"errors": ["no result file"]}
    _res.update(returncode=_rc, sec=round(time.time() - _t, 1), configs_in_vllm=_installed, gpu_released=_free,
                log_evidence=log_evidence(_logp.read_text(errors="replace")))
    BENCH[_name] = _res
    print(_name, "rc", _rc, "| load", _res.get("load_sec"), "| short tok/s", (_res.get("workloads", {}).get("short") or {}).get("tok_s"),
          "| long tok/s", (_res.get("workloads", {}).get("long") or {}).get("tok_s"), "| evidence", _res["log_evidence"])
    RESULTS["stages"]["bench"] = BENCH; save_results()
set_configs(False)            # leave vLLM as it was
RESULTS["stages"]["bench"] = BENCH; save_results()


In [ ]:
# ==================================================================== CELL 5 — stage 4: report
def _w(v, w, k):
    return ((BENCH.get(v) or {}).get("workloads") or {}).get(w, {}).get(k)

def _prefix(a, b):
    n = 0
    for x, y in zip(a, b):
        if x != y: break
        n += 1
    return n

base = {w: _w("default", w, "tok_s") for w in ("short", "long")}
rows, summary = [], {}
for v in ("default", "tuned", "marlin"):
    b = BENCH.get(v) or {}
    if "skipped" in b:
        rows.append(f"| {v} | skipped: {b['skipped']} |||||||"); continue
    sp = {w: (round(_w(v, w, "tok_s") / base[w], 3) if base.get(w) and _w(v, w, "tok_s") else None) for w in ("short", "long")}
    g0, g1 = (BENCH.get("default") or {}).get("greedy_ids") or [], b.get("greedy_ids") or []
    agree = [_prefix(x, y) for x, y in zip(g0, g1)] if v != "default" else None
    summary[v] = {"short_tok_s": _w(v, "short", "tok_s"), "long_tok_s": _w(v, "long", "tok_s"), "speedup_vs_default": sp,
                  "accept_short": _w(v, "short", "accept_rate"), "accept_long": _w(v, "long", "accept_rate"),
                  "preemptions": (_w(v, "short", "preemptions") or 0) + (_w(v, "long", "preemptions") or 0),
                  "greedy_common_prefix_vs_default": agree, "load_sec": b.get("load_sec"), "errors": b.get("errors"),
                  "log_evidence": b.get("log_evidence")}
    rows.append(f"| {v} | {b.get('load_sec')} | {_w(v, 'short', 'tok_s')} | {sp['short']} | {_w(v, 'long', 'tok_s')} | {sp['long']} | "
                f"{_w(v, 'short', 'accept_rate')} / {_w(v, 'long', 'accept_rate')} | {agree} |")
RESULTS["summary"] = summary
tune_rows = []
for k, d in (RESULTS["stages"].get("tune", {}).get("timings") or {}).items():
    t = d.get("timings", {})
    tune_rows.append(f"| {k} | " + " | ".join(str((t.get(str(m)) or {}).get("speedup")) for m in (16, 64, 128, 512, 4096)) +
                     f" | {'yes' if d.get('complete') else 'partial'} |")
md = ["# E12-K1 FP8 kernel probe: results", "",
      f"Started {RESULTS['started_utc']}, total {round((time.time() - T0) / 60, 1)} min, dryrun {DRYRUN}.", "",
      "## Engine decode speed (R5 engine settings; 16 requests)", "",
      "| Variant | Load s | Short tok/s (2–10k ctx) | × default | Long tok/s (~36k ctx) | × default | MTP accept short / long | Greedy common prefix vs default (4 × 192) |",
      "|---|---|---|---|---|---|---|---|"] + rows + ["",
      "Kernel-path evidence from each log (default warnings / tuned config used / Marlin mentions):", ""] + \
     [f"- {v}: {json.dumps((BENCH.get(v) or {}).get('log_evidence'))}" for v in ("default", "tuned", "marlin")] + ["",
      "## Kernel tuning speedup over the default config (per GEMM shape, at M tokens)", "",
      "| Shape | M=16 | M=64 | M=128 | M=512 | M=4096 | Complete |", "|---|---|---|---|---|---|---|"] + tune_rows + ["",
      "Tuned config files (ship these): `k1/configs_l4/`. Copy them into vLLM's "
      "`model_executor/layers/quantization/utils/configs/` before the engine starts."]
(OUT / "report.md").write_text("\n".join(md) + "\n", encoding="utf-8")
RESULTS["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()); save_results()
print("\n".join(md))
